In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
# Load Titanic dataset for classification
df = pd.read_csv("../data/titanic.csv")
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Data Preprocessing

In [ ]:
class DataPreprocessor:
    """
    Simple preprocessor for ML datasets.
    Handles missing values, encoding, and scaling.
    """
    
    def __init__(self):
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.numeric_imputer = SimpleImputer(strategy='median')
        self.categorical_imputer = SimpleImputer(strategy='most_frequent')
    
    def fit_transform(self, df: pd.DataFrame, target_col: str):
        """
        Preprocess data: handle missing values, encode categoricals, scale numerics.
        """
        df = df.copy()
        
        # Separate features and target
        y = df[target_col].copy()
        X = df.drop(columns=[target_col])
        
        # Identify column types
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        
        print(f"Numeric columns: {numeric_cols}")
        print(f"Categorical columns: {categorical_cols}")
        
        # Handle missing values - numeric
        if numeric_cols:
            X[numeric_cols] = self.numeric_imputer.fit_transform(X[numeric_cols])
        
        # Handle missing values - categorical
        if categorical_cols:
            X[categorical_cols] = self.categorical_imputer.fit_transform(X[categorical_cols])
            
            # Encode categoricals
            for col in categorical_cols:
                le = LabelEncoder()
                X[col] = le.fit_transform(X[col].astype(str))
                self.label_encoders[col] = le
        
        # Scale numeric features
        if numeric_cols:
            X[numeric_cols] = self.scaler.fit_transform(X[numeric_cols])
        
        # Handle target if it's categorical
        if y.dtype == 'object':
            le = LabelEncoder()
            y = pd.Series(le.fit_transform(y), name=target_col)
            self.label_encoders[target_col] = le
        
        print(f"\n✅ Preprocessing complete!")
        print(f"   X shape: {X.shape}")
        print(f"   y shape: {y.shape}")
        
        return X, y

In [ ]:
# Preprocess Titanic data
preprocessor = DataPreprocessor()
X, y = preprocessor.fit_transform(df, target_col="Survived")

## 3. Train/Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # For classification, stratify by target
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTarget distribution:")
print(f"  Train: {y_train.value_counts().to_dict()}")
print(f"  Test: {y_test.value_counts().to_dict()}")

## 4. Classification Models

In [ ]:
def train_classification_models(X_train, X_test, y_train, y_test):
    """
    Train multiple classification models and compare performance.
    """
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    }
    
    results = []
    trained_models = {}
    
    print("="*60)
    print("🏋️ TRAINING CLASSIFICATION MODELS")
    print("="*60)
    
    for name, model in models.items():
        print(f"\n📌 Training {name}...")
        
        # Train
        model.fit(X_train, y_train)
        trained_models[name] = model
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        # Cross-validation score
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
        
        results.append({
            'Model': name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1,
            'CV Mean': cv_scores.mean(),
            'CV Std': cv_scores.std()
        })
        
        print(f"   Accuracy: {accuracy:.4f}")
        print(f"   F1 Score: {f1:.4f}")
        print(f"   CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    return pd.DataFrame(results), trained_models

In [ ]:
# Train models
results_df, trained_models = train_classification_models(X_train, X_test, y_train, y_test)

In [ ]:
# View results
print("\n" + "="*60)
print("📊 MODEL COMPARISON")
print("="*60)
results_df.sort_values('F1 Score', ascending=False)

## 5. Best Model Analysis

In [ ]:
# Get best model
best_model_name = results_df.loc[results_df['F1 Score'].idxmax(), 'Model']
best_model = trained_models[best_model_name]

print(f"🏆 Best Model: {best_model_name}")

# Detailed classification report
y_pred = best_model.predict(X_test)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n🔢 Confusion Matrix:")
print(pd.DataFrame(cm, 
                   index=['Actual 0', 'Actual 1'],
                   columns=['Pred 0', 'Pred 1']))

## 6. Regression Example

In [ ]:
def train_regression_models(X_train, X_test, y_train, y_test):
    """
    Train multiple regression models and compare performance.
    """
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=0.1),
        'Decision Tree': DecisionTreeRegressor(random_state=42),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    }
    
    results = []
    
    print("="*60)
    print("🏋️ TRAINING REGRESSION MODELS")
    print("="*60)
    
    for name, model in models.items():
        print(f"\n📌 Training {name}...")
        
        # Train
        model.fit(X_train, y_train)
        
        # Predict
        y_pred = model.predict(X_test)
        
        # Metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
        
        results.append({
            'Model': name,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2,
            'CV Mean': cv_scores.mean(),
            'CV Std': cv_scores.std()
        })
        
        print(f"   RMSE: {rmse:.4f}")
        print(f"   R2: {r2:.4f}")
    
    return pd.DataFrame(results)

In [ ]:
# Load demo_sales for regression
df_sales = pd.read_csv("../data/demo_sales.csv")
print(f"Sales Dataset shape: {df_sales.shape}")
df_sales.head()

In [ ]:
# Preprocess sales data
preprocessor_sales = DataPreprocessor()
X_sales, y_sales = preprocessor_sales.fit_transform(df_sales, target_col="Revenue")

# Split
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sales, y_sales, test_size=0.2, random_state=42
)

# Train regression models
regression_results = train_regression_models(X_train_s, X_test_s, y_train_s, y_test_s)

In [ ]:
# View regression results
print("\n" + "="*60)
print("📊 REGRESSION MODEL COMPARISON")
print("="*60)
regression_results.sort_values('R2', ascending=False)

## 7. Feature Importance

In [ ]:
# Get feature importance from Random Forest
rf_model = trained_models['Random Forest']

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("🌲 Random Forest Feature Importance (Titanic):")
print(feature_importance.head(10))

## ✅ Summary

This module provides:

**1. DataPreprocessor**
- Handle missing values (median for numeric, mode for categorical)
- Label encoding for categorical features
- Feature scaling with StandardScaler

**2. Classification Models**
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting

**3. Regression Models**
- Linear Regression, Ridge, Lasso
- Decision Tree, Random Forest, Gradient Boosting

**4. Evaluation**
- Cross-validation
- Classification metrics (accuracy, precision, recall, F1)
- Regression metrics (RMSE, MAE, R²)
- Feature importance analysis